In [1]:
# CELL 1 — Get dataset and install packages
!git clone https://github.com/rafalposwiata/depression-detection-lt-edi-2022.git
!pip install contractions emoji -q

fatal: destination path 'depression-detection-lt-edi-2022' already exists and is not an empty directory.


In [2]:
# CELL 2 — Load data and collapse labels to binary
import pandas as pd

train = pd.read_csv('depression-detection-lt-edi-2022/data/preprocessed_dataset/train.csv')
dev   = pd.read_csv('depression-detection-lt-edi-2022/data/preprocessed_dataset/dev.csv')
test  = pd.read_csv('depression-detection-lt-edi-2022/data/preprocessed_dataset/test.csv')

# 0 = severe, 1 = moderate -> depressed | 2 = not depressed
label_map = {0: 'depressed', 1: 'depressed', 2: 'not depressed'}
for df in [train, dev, test]:
    df['labels'] = df['labels'].replace(label_map)

print('Label distribution in train:')
print(train['labels'].value_counts())

Label distribution in train:
labels
depressed        3751
not depressed    2255
Name: count, dtype: int64


In [3]:
# Cell 3: Data Cleaning

import re
import contractions
import emoji
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

# Custom stopwords preserving psycholinguistically important words
# Following Tadesse et al. (2019), Trifan et al. (2020), Low et al. (2020)
stop_words = set(stopwords.words('english'))
keep_words = {
    'i', 'me', 'myself', 'mine',           # first person pronouns
    'no', 'not', 'never', 'nothing',        # negation words
    'nobody', 'nowhere', 'nor',             # negation words
    'always', 'every', 'everything',        # absolutist words
    'everyone', 'must', 'completely',       # absolutist words
    'totally', 'absolutely'                 # absolutist words
}
stop_words = stop_words - keep_words

reddit_slang = {
    r'\btldr\b': '', r'\btl;dr\b': '',
    r'\bbf\b': 'boyfriend', r'\bgf\b': 'girlfriend',
    r'\bdh\b': 'husband',
    r'\bmil\b': 'mother in law', r'\bfil\b': 'father in law',
    r'\bsil\b': 'sister in law', r'\bbil\b': 'brother in law',
    r'\bimo\b': 'in my opinion', r'\bimho\b': 'in my opinion',
    r'\bidk\b': 'i do not know', r'\bidc\b': 'i do not care',
    r'\bngl\b': 'not going to lie', r'\bfml\b': 'my life is bad',
    r'\bsmh\b': 'shaking my head', r'\bntm\b': 'not to mention',
}

# VERSION 1 — text_clean
# Used for TF-IDF and classical models
# Stopwords removed, special chars removed, slang expanded

def clean_text(text):
    text = str(text)
    text = re.sub(r'\b(train|dev|test)_pid_\d+\b', '', text)          #remove Reddit post IDs e.g. train_pid_4429
    text = re.sub(r'\b\d{1,2}[FfMmNnBb]{1,2}\b', '', text)           #remove age/gender markers e.g. 23F 24M
    text = re.sub(r'r/\w+', '', text)                                  #remove subreddit references e.g. r/depression
    text = re.sub(r'http\S+|www\S+', '', text)                         # remove URLs
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)                         # remove punctuation, numbers, special chars
    text = re.sub(r'&amp;', 'and', text)                               # fix HTML &amp;
    text = re.sub(r'&gt;|&lt;|&#x200B;', '', text)                    # fix HTML invisible spaces
    text = emoji.demojize(text)                                        # convert emojis to text
    text = contractions.fix(text)                                      # expand contractions
    text = text.lower()
    for pattern, replacement in reddit_slang.items():
        text = re.sub(pattern, replacement, text)                      # expand Reddit slang e.g. bf to boyfriend
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)                       # collapse repeated chars insome words
    words = text.split()
    words = [w for w in words if w not in stop_words and (len(w) > 1 or w == 'i')]
    text = re.sub(r'\s+', ' ', ' '.join(words)).strip()               # clean extra whitespace
    return text


# VERSION 2: text_minimal
# version for necessary human readable error analysis
# Only adds to the text.. expands contractions, explains emojis
# Removes post IDs and URLs, nothing else
def clean_text_minimal(text):
    text = str(text)
    text = re.sub(r'\b(train|dev|test)_pid_\d+\b', '', text)  # remove post IDs
    text = re.sub(r'http\S+|www\S+', '', text)                 # remove URLs
    text = re.sub(r'&amp;', 'and', text)                       # fix HTML artifacts
    text = re.sub(r'&gt;|&lt;|&#x200B;', '', text)             # fix HTML artifacts
    text = emoji.demojize(text)                                # explain emojis
    text = contractions.fix(text)                              # expand contractions
    text = re.sub(r'\s+', ' ', text).strip()                   # clean whitespace
    return text

for df in [train, dev, test]:
    df['text_clean']   = df['text'].apply(clean_text)
    df['text_minimal'] = df['text'].apply(clean_text_minimal)

print('Cleaning done.')
print('\ntext_clean sample:')
print(train['text_clean'].iloc[0][:200])
print('\ntext_minimal sample:')
print(train['text_minimal'].iloc[0][:200])

Cleaning done.

text_clean sample:
point genuinely curious sort shit throws way actually funny i used end every previous year okay bad surely get much worse go rock bottom get fucked unimaginable ways next year point i mostly dissociat

text_minimal sample:
At this point just genuinely curious what sort of shit 2020 throws my way : It is actually funny. I used to end every previous year with "Okay, that was bad, but surely it cannot get much worse!! You 


In [4]:
# CELL 4 — Absolutist word score
# Based on Trifan et al. (2020) citing Al-Mosaiwi and Johnstone (2018)
# Ratio of absolutist words to total post length

absolutist_words = [
    'always', 'never', 'everything', 'nothing', 'everyone',
    'no one', 'nobody', 'completely', 'totally', 'entirely',
    'whole', 'must', 'definitely', 'absolutely', 'constant',
    'constantly', 'every', 'forever', 'impossible', 'perfect',
    'perfectly', 'utterly', 'purely'
]

def absolutist_score(text):
    words = str(text).lower().split()
    count = sum(1 for w in words if w in absolutist_words)
    return count / len(words) if len(words) > 0 else 0

for df in [train, dev, test]:
    df['absolutist_score'] = df['text_clean'].apply(absolutist_score)

print('Average absolutist score by class:')
print(train.groupby('labels')['absolutist_score'].mean().round(4))

Average absolutist score by class:
labels
depressed        0.0215
not depressed    0.0196
Name: absolutist_score, dtype: float64


In [5]:
# CELL 5 : Save final files
# Keep only clean text, label, and absolutist score
# Drop pid and original text
cols = ['text_clean', 'text_minimal', 'labels', 'absolutist_score']

train[cols].to_csv('train_clean.csv', index=False)
dev[cols].to_csv('dev_clean.csv', index=False)
test[cols].to_csv('test_clean.csv', index=False)

print('Saved successfully.')
print('Columns:', cols)
print('Train shape:', train[cols].shape)
print('\nSample first 3 rows:')
print(train[cols][['text_minimal', 'labels']].head(3))



Saved successfully.
Columns: ['text_clean', 'text_minimal', 'labels', 'absolutist_score']
Train shape: (6006, 4)

Sample first 3 rows:
                                        text_minimal     labels
0  At this point just genuinely curious what sort...  depressed
1  I have literally tried everything.... : I am s...  depressed
2  I am really struggling : So I do not know how ...  depressed
